## 1. Theory

### 1.1 Why do we need a Pipeline?

Machine-learning data often require several preprocessing steps before a model can be trained. For example:

```text
Raw Data
   ↓
Missing-value handling
   ↓
Scaling numerical features
   ↓
Encoding categorical features
   ↓
Machine-Learning Model
```

A scikit-learn `Pipeline` combines these operations into a single object.

### Main advantage

The preprocessing steps are fitted **only on the training portion** during cross-validation. This helps prevent **data leakage**.

A typical pipeline is:

```python
Pipeline([
    ('preprocessor', preprocessor),
    ('model', classifier)
])
```


### 1.2 Numerical vs. Categorical Features

**Numerical features** contain quantitative values, e.g. `age` and `fare`.

**Categorical features** represent groups or labels, e.g. `color`, `embarked`, and `pclass`.

Numerical features can be standardized using:

`StandardScaler`

Categorical features can be converted into binary indicator columns using:

`BinaryEncoder`, `LabelEncoder`, and `OneHotEncoder`

**BinaryEncoder Example:**

```text
is_in_danger
----
yes
no
yes
```

can become:

```text
is_in_danger
1
0
1
```
**LabelEncoder Example:**

```text
grade
----
A
B
A
C
```

can become:

```text
grade
----
0
1
0
2
```


**OneHotEncoder Example:**

```text
color
----
blue
red
blue
green
```

can become:

```text
color_red   color_blue   color_green
0           1            0
1           0            0
0           1            0
0           0            1
```

equivalent to:

```text
color_red   color_blue
0           1
1           0
0           1
0           0
```

### 1.3 ColumnTransformer

Different columns require different preprocessing. `ColumnTransformer` allows us to apply one transformation to numerical columns and another transformation to categorical columns.

Conceptually:

```text
                    ┌── Numerical ── StandardScaler ──┐
Input Data ─────────┤                                 ├── Combined Features
                    └── Categorical   ──  Encoder   ──┘
```

The resulting `ColumnTransformer` can then be placed inside a `Pipeline`.


### 1.4 k-Fold Cross-Validation

In **k-fold cross-validation**, the dataset is divided into `k` parts (folds).

For each round, one fold is used for validation and the remaining folds are used for training.

For `k = 5`:

```text
Round 1:  [Validation] [Train] [Train] [Train] [Train]
Round 2:  [Train] [Validation] [Train] [Train] [Train]
Round 3:  [Train] [Train] [Validation] [Train] [Train]
Round 4:  [Train] [Train] [Train] [Validation] [Train]
Round 5:  [Train] [Train] [Train] [Train] [Validation]
```

The final performance is usually reported as the mean and standard deviation of the fold scores.


### 1.5 Statistical Tests for Model Comparison in Machine Learning

When comparing two Machine Learning models, statistical tests help us determine whether one model is genuinely better than another or whether the observed performance difference could be due to random variation.

For example, suppose we train two models to classify emails as spam or not spam.

- Model A: Accuracy = 92%
- Model B: Accuracy = 94%

Model B has higher accuracy, but is the 2% improvement statistically significant? Statistical testing helps answer this question.

#### i. How does model comparison work?

```
Model A Accuracy = 92%
Model B Accuracy = 94%
```

            ⇓

```
Statistical hypothesis test
(Evaluate the observed performance difference)
```

            ⇓
```
Significant → Evidence of a real performance difference under the test assumptions.
Not significant → Insufficient evidence that performance differs.
```
## 2. Which Statistical Tests Are Used?

The statistical test you choose depends on how you compare your machine learning models.

| Statistical test | When to use it |
| --- | --- |
| Paired t-test | Compare two models' scores on the same datasets or experiments. |
| Wilcoxon signed-rank test | Compare two models when their scores are paired, but the data do not meet the assumptions of a paired t-test. |
| McNemar's test | Compare two classification models using their predictions on the same test data. |
| 5×2 Cross-Validation paired t-test | Compare two models using a specific method that repeats 2-fold cross-validation five times. |
| Friedman test | Compare three or more machine learning models across multiple datasets. |

Important: Choosing a statistical test depends not only on the number of models but also on how the models are tested and what type of data you have.

## 3. Example: Comparing two models using a paired t-test

Suppose we evaluate two models on five different folds. Each value below represents the accuracy measured on one fold.

| Dataset | Model A | Model B | Difference (B − A) |
| --- | --- | --- | --- |
| 1 | 80% | 84% | +4% |
| 2 | 82% | 85% | +3% |
| 3 | 79% | 81% | +2% |
| 4 | 85% | 86% | +1% |
| 5 | 81% | 85% | +4% |
| Mean | 81.4% | 84.2% | +2.8% |

Model B performs better on all five folds. But we still need to quantify the uncertainty in the observed difference.

### Step 1: Define the hypotheses

- Null hypothesis ($H_0$): The population mean difference in accuracy is zero.
- Alternative hypothesis ($H_1$): The population mean difference in accuracy is not zero.

We choose a significance level of $\alpha = 0.05$.

### Step 2: Run the test in Python


```python
from scipy.stats import ttest_rel

model_a = [80, 82, 79, 85, 81]
model_b = [84, 85, 81, 86, 85]

t_stat, p_value = ttest_rel(model_a, model_b)

print("T-statistic:", t_stat)
print("P-value:", p_value)

if p_value < 0.05:
    print("Significant performance difference")
else:
    print("Insufficient evidence of a difference")
```

For these example values, the paired t-test gives approximately:

- Mean difference: 2.8 percentage points
- t-statistic: 5.25
- p-value: 0.006

### Step 3: Interpret the result

Since $0.006 < 0.05$, we reject the null hypothesis.

Conclusion: There is statistically significant evidence of a difference in mean accuracy, with Model B performing better in these observations.



## 4. Example: Comparing prediction errors using a paired t-test

Suppose we evaluate a baseline model and an improved model on five observations. For each observation, we calculate the absolute prediction error, which measures how far the prediction is from the true value.

The baseline model produces predictions `pred`, while the improved model produces predictions `preds`. Our goal is to determine whether the improved model significantly reduces the prediction error.

| Observation | True value (`y_true_seq`) | Baseline prediction (`pred`) | Improved prediction (`preds`) | Baseline error (`base_err`) | Improved error (`err`) |
| --- | --- | --- | --- | --- | --- |
| 1 | 10 | 8 | 9 | 2 | 1 |
| 2 | 20 | 22 | 19 | 2 | 1 |
| 3 | 30 | 27 | 31 | 3 | 1 |
| 4 | 40 | 43 | 39 | 3 | 1 |
| 5 | 50 | 46 | 49 | 4 | 1 |
| Mean | 30 | 29.2 | 29.4 | 2.8 | 1.0 |

The improved model has a lower mean absolute error (MAE) than the baseline model:

- Baseline MAE: $2.8$
- Improved model MAE: $1.0$
- Mean error reduction: $2.8 - 1.0 = 1.8$

However, we still need to determine whether the observed reduction is statistically significant.

### Step 1: Define the hypotheses

Let $e_i$ denote the absolute error of the improved model and $b_i$ denote the absolute error of the baseline model for observation $i$.

We define the paired differences as

$$
d_i = e_i-b_i
$$

The hypotheses for a two-sided paired t-test are:

- Null hypothesis ($H_0$): The population mean difference in absolute prediction errors is zero, $\mu_d=0$.
- Alternative hypothesis ($H_1$): The population mean difference in absolute prediction errors is not zero, $\mu_d\neq 0$.

We choose a significance level of $\alpha=0.05$.

### Step 2: Run the test in Python

The following code implements the same calculations as your original code section.

```python
import numpy as np
from scipy import stats

y_true_seq = np.array([10, 20, 30, 40, 50])

# Baseline model predictions
pred = np.array([8, 22, 27, 43, 46])

# Improved model predictions
preds = np.array([9, 19, 31, 39, 49])

# Calculate absolute prediction errors
err = np.abs(y_true_seq - preds)
base_err = np.abs(y_true_seq - pred)

# Perform the paired t-test
t_stat, t_p = stats.ttest_rel(err, base_err)

print("Baseline errors:", base_err)
print("Improved errors:", err)
print("Baseline MAE:", np.mean(base_err))
print("Improved MAE:", np.mean(err))
print("T-statistic:", t_stat)
print("P-value:", t_p)

if t_p < 0.05:
    print("Significant difference in prediction errors")
else:
    print("Insufficient evidence of a difference")
```

For these example values, the absolute errors are:

text
```
Baseline errors: [2 2 3 3 4]
Improved errors: [1 1 1 1 1]
Baseline MAE: 2.8
Improved MAE: 1.0
```

The paired t-test results are approximately:

- Mean difference (`err - base_err`): $-1.8$
- t-statistic: $-5.40$
- p-value: $0.0057$

### Step 3: Interpret the result

Since $0.0057 < 0.05$, we reject the null hypothesis.

Conclusion: There is statistically significant evidence of a difference in the mean absolute prediction errors between the two models. Because the improved model has a lower mean absolute error, the results support better predictive accuracy for the improved model on these observations.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import ttest_rel


## 2. Load the Dataset

Run the following cell. Then inspect the columns and identify which variables are numerical and categorical.

In [ ]:
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

print(df.shape)
display(df.head())
display(df.dtypes)

### Task 1
Inspect the dataset and identify:

- Target variable
- Numerical features
- Categorical features

In [ ]:
# Keep only the variables needed for this lab.
# TODO: Complete this list.

features = [
    'Pclass',
    'Sex',
    'Age',
]

# target = ...

X = df[features]
y = df[target]

display(X.head())
print('Target distribution:')
print(y.value_counts())

## 3. Build the Preprocessor

We will use:

- `SimpleImputer(strategy='median')` + `StandardScaler()` for numerical features.
- `SimpleImputer(strategy='most_frequent')` + `OneHotEncoder(handle_unknown='ignore')` for categorical features.

### Task 2 — Complete the Function

Complete the function below. **Do not write the preprocessing directly into the model.** The goal is to create a reusable `ColumnTransformer`.

In [ ]:
def build_preprocessor(numerical_features, categorical_features):
    """Create the numerical + categorical preprocessing pipeline."""

    # TODO 1: create numerical preprocessing
    # numerical_transformer = Pipeline([...])

    # TODO 2: create categorical preprocessing
    # categorical_transformer = Pipeline([...])

    # TODO 3: combine both using ColumnTransformer
    # preprocessor = ColumnTransformer([...])

    pass

In [ ]:
# Complete these two lists

numerical_features = [
    # TODO
]

categorical_features = [
    # TODO
]

preprocessor = build_preprocessor(numerical_features, categorical_features)
preprocessor

## 4. Build the Machine-Learning Pipeline

The complete workflow should be:

```text
X --> Preprocessing --> Machine-Learning Model --> Prediction
```

### Task 3 — Complete the Function

Create a function that receives a preprocessor and a model and returns a scikit-learn `Pipeline`.

In [ ]:
def build_pipeline(preprocessor, model):
    """Return a preprocessing + model Pipeline."""

    # TODO: create and return Pipeline([...])
    pass

## 5. Compare Multiple Algorithms

We will compare three classifiers:

1. Logistic Regression
2. Decision Tree
3. Random Forest

### Task 4
Create the three model objects and store them in a dictionary.

In [ ]:
models = {
    # TODO: 'Logistic Regression': LogisticRegression(...),
    # TODO: 'Decision Tree': DecisionTreeClassifier(...),
    # TODO: 'Random Forest': RandomForestClassifier(...)
}

models

## 6. 5-Fold Cross-Validation

Use **StratifiedKFold** because this is a classification problem. `shuffle=True` makes the folds randomized, while `random_state` makes the experiment reproducible.

### Task 5 — Complete the Function

For each model:
1. Build the complete Pipeline.
2. Calculate 5-fold accuracy scores.
3. Return the scores.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_model(model, X, y, preprocessor, cv):
    """Return the five cross-validation accuracy scores."""

    # TODO 1: build the complete pipeline
    # pipeline = build_pipeline(preprocessor, model)

    # TODO 2: calculate cross-validation scores
    # scores = cross_val_score(...)

    # TODO 3: return scores
    pass

In [ ]:
# Complete this cell

results = {}

for name, model in models.items():
    # TODO: call evaluate_model(...)
    # results[name] = ...
    pass

for name, scores in results.items():
    print(f'{name}: {scores}')


## 7. Summarize the Results

Calculate the mean and standard deviation of the five accuracy scores for every model.

### Task 6

In [ ]:
# TODO: Create a DataFrame containing:
# Model | Mean Accuracy | Std Accuracy

summary = []

for name, scores in results.items():
    summary.append({
        'Model': name,
        # 'Mean Accuracy': ...,
        # 'Std Accuracy': ...
    })

results_df = pd.DataFrame(summary)
results_df

## 8. Statistical Test: Paired t-test

Now compare two models using their **five paired cross-validation scores**.

For this lab, compare Logistic Regression and Random Forest.

### Task 7

Complete the code below using `ttest_rel()`.

Interpret the result using α = 0.05.


In [ ]:
# TODO: obtain the two arrays of CV scores
logistic_scores = results['Logistic Regression']
random_forest_scores = results['Random Forest']

# TODO: perform the paired t-test
# t_stat, p_value = ttest_rel(logistic_scores, random_forest_scores)

# print(t_stat, p_value)
